<a href="https://colab.research.google.com/github/prince9939367489/Diffusion-Model-Based-Cancelable-Biometric-Templates/blob/main/Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cancelable Biometric Template Benchmark

This notebook compares **Index-of-Max (IoM) hashing**, **BioHashing**, and **random projection** on features derived from an eye-image dataset. It then evaluates four classical classifiers: Random Forest, RBF SVM, KNN, and XGBoost.

> **Scope note:** The current notebook does not implement a diffusion model, despite the repository name. It is a cancelable-template classification experiment built around an ImageNet-pretrained EfficientNetB0 feature pipeline.

## Experiment pipeline

`class folders → 224 × 224 images → EfficientNetB0 pipeline → cancelable transform → tuned classifier → weighted classification metrics`


## Data and reproducibility

The dataset is not distributed with this repository. The default Colab location is `/content/drive/MyDrive/Eye dataset`, with one subfolder per class. To use another location, edit `DATASET_PATH_OVERRIDE` in the next cell or set the `EYE_DATASET_PATH` environment variable before running the experiment cell.

The code below sorts class folders and image files, centralizes the random seed, and requests deterministic TensorFlow operations where supported. Exact reproduction still depends on the unavailable dataset, package versions, hardware, and TensorFlow kernels.

> The output attached to the experiment cell is preserved from the original committed run. It was not regenerated after these portability and reproducibility edits.


In [ ]:
# Optional path override. Leave as None to use EYE_DATASET_PATH or the Colab default.
DATASET_PATH_OVERRIDE = None
# Example: DATASET_PATH_OVERRIDE = "/content/drive/MyDrive/Eye dataset"

In [ ]:
# =========================
# INSTALL LIBRARIES
# =========================
!pip install tensorflow scikit-learn opencv-python xgboost

# =========================
# IMPORT LIBRARIES
# =========================
import os
from pathlib import Path

SEED = 42
os.environ.setdefault("PYTHONHASHSEED", str(SEED))
os.environ.setdefault("TF_DETERMINISTIC_OPS", "1")

import cv2
import numpy as np
import tensorflow as tf

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except (AttributeError, RuntimeError) as error:
    print(f"TensorFlow deterministic operations are unavailable: {error}")

# =========================
# DATASET CONFIGURATION
# =========================
DEFAULT_DATASET_PATH = Path("/content/drive/MyDrive/Eye dataset")
dataset_path_override = globals().get("DATASET_PATH_OVERRIDE")
configured_path = dataset_path_override or os.environ.get(
    "EYE_DATASET_PATH", str(DEFAULT_DATASET_PATH)
)
data_path = Path(configured_path).expanduser()

# Mount Drive only when the selected path points into the Colab Drive mount.
if str(data_path).startswith("/content/drive/"):
    try:
        from google.colab import drive
    except ImportError as error:
        raise RuntimeError(
            "The default dataset path requires Google Colab. Set EYE_DATASET_PATH "
            "to run elsewhere."
        ) from error
    drive.mount("/content/drive")

if not data_path.is_dir():
    raise FileNotFoundError(
        f"Dataset directory not found: {data_path}. "
        "Set EYE_DATASET_PATH or place the dataset at the default Colab path."
    )

print("Dataset path:", data_path)

# =========================
# LOAD DATA
# =========================
IMG_SIZE = 224

def load_data(data_dir):
    data_dir = Path(data_dir)
    X, y = [], []
    class_paths = sorted(path for path in data_dir.iterdir() if path.is_dir())

    if len(class_paths) < 2:
        raise ValueError("Expected at least two class folders in the dataset directory.")

    for label, class_path in enumerate(class_paths):
        image_paths = sorted(path for path in class_path.iterdir() if path.is_file())
        for image_path in image_paths:
            img = cv2.imread(str(image_path))
            if img is None:
                print(f"Skipping unreadable image: {image_path}")
                continue

            try:
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            except cv2.error as error:
                print(f"Skipping invalid image {image_path}: {error}")
                continue

            X.append(img)
            y.append(label)

    if not X:
        raise ValueError(f"No readable images were found in {data_dir}.")

    class_names = [path.name for path in class_paths]
    return np.asarray(X), np.asarray(y), class_names

X, y, class_names = load_data(data_path)

print("Class folders:", class_names)
print("Total Images:", len(X))
print("Total Classes:", len(class_names))

# =========================
# PREPROCESS
# =========================
X = preprocess_input(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# =========================
# FEATURE EXTRACTION
# =========================
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
# This projection head is intentionally left untrained to retain the original method.
x = Dense(512, activation='relu')(x)
x = Dropout(0.4)(x)

feature_model = Model(inputs=base_model.input, outputs=x)

print("Extracting Features...")
train_features = feature_model.predict(X_train, batch_size=32)
test_features = feature_model.predict(X_test, batch_size=32)

# =========================
# CANCELABLE BIOMETRIC METHODS
# =========================
def projection_matrix(input_dim, output_dim, seed=SEED):
    # RandomState preserves the matrix generation used by the original notebook.
    return np.random.RandomState(seed).randn(input_dim, output_dim)

def iom_hash(features, matrix, k=5):
    projected = np.dot(features, matrix)
    return np.argsort(projected, axis=1)[:, -k:]

def biohashing(features, matrix):
    bio = np.dot(features, matrix)
    return (bio > 0).astype(int)

def random_projection(features, matrix):
    return np.dot(features, matrix)

feature_dim = train_features.shape[1]
iom_matrix = projection_matrix(feature_dim, 256)
biohash_matrix = projection_matrix(feature_dim, feature_dim)
rp_matrix = projection_matrix(feature_dim, 128)

# Reuse each matrix for train and test so both splits share one transform.
cb_train = {
    "IoM": iom_hash(train_features, iom_matrix),
    "BioHash": biohashing(train_features, biohash_matrix),
    "RP": random_projection(train_features, rp_matrix)
}

cb_test = {
    "IoM": iom_hash(test_features, iom_matrix),
    "BioHash": biohashing(test_features, biohash_matrix),
    "RP": random_projection(test_features, rp_matrix)
}

# =========================
# EVALUATION FUNCTION
# =========================
def evaluate_model(name, model, X_tr, X_te):
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    print(f"\n{name}:")
    if hasattr(model, "best_params_"):
        print("Best Params:", model.best_params_)
    print("Accuracy :", acc)
    print("Precision:", prec)
    print("Recall   :", rec)
    print("F1 Score :", f1)

# =========================
# MAIN LOOP
# =========================
for cb_name in cb_train:
    print(f"\n===== CB Technique: {cb_name} =====")

    X_tr = cb_train[cb_name].reshape(cb_train[cb_name].shape[0], -1)
    X_te = cb_test[cb_name].reshape(cb_test[cb_name].shape[0], -1)

    #  RandomForest
    rf = GridSearchCV(
        RandomForestClassifier(random_state=SEED),
        {
            'n_estimators': [100, 200],
            'max_depth': [None, 10],
        },
        cv=3, n_jobs=-1
    )
    evaluate_model("RandomForest (Tuned)", rf, X_tr, X_te)

    #  SVM
    svm = GridSearchCV(
        SVC(),
        {
            'C': [1, 10],
            'gamma': ['scale', 0.01],
            'kernel': ['rbf']
        },
        cv=3, n_jobs=-1
    )
    evaluate_model("SVM (Tuned)", svm, X_tr, X_te)

    #  KNN
    knn = GridSearchCV(
        KNeighborsClassifier(),
        {
            'n_neighbors': [3, 5],
        },
        cv=3, n_jobs=-1
    )
    evaluate_model("KNN (Tuned)", knn, X_tr, X_te)

    #  XGBoost
    xgb = RandomizedSearchCV(
        XGBClassifier(
            objective='multi:softmax',
            num_class=len(class_names),
            eval_metric='mlogloss',
            random_state=SEED
        ),
        {
            'n_estimators': [100, 200],
            'max_depth': [3, 5],
            'learning_rate': [0.01, 0.1]
        },
        n_iter=5,
        random_state=SEED,
        cv=3,
        n_jobs=-1
    )
    evaluate_model("XGBoost (Tuned)", xgb, X_tr, X_te)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total Images: 14161
Total Classes: 4
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Extracting Features...
354/354 ━━━━━━━━━━━━━━━━━━━━ 899s 3s/step
89/89 ━━━━━━━━━━━━━━━━━━━━ 237s 3s/step

===== CB Technique: IoM =====

RandomForest (Tuned):
Best Params: {'max_depth': None, 'n_estimators': 100}
Accuracy : 0.7511471937875044
Precision: 0.7507202422633473
Recall   : 0.7511471937875044
F1 Score : 0.7502661318261603

SVM (Tuned):
Best Params: {'C': 10, 'gamma': 0.01, 'kernel': 'rbf'}
Accuracy : 0.6392516766678433
Precision: 0.6691036348101985
Recall   : 0.6392516766678433
F1 Score : 0.6264191048001166

KNN (Tuned):
Best Params: {'n_neighbors': 3}
Accuracy : 0.616307800917755
Precision: 0.6192660199484686
Recall   : 0.616307800917755
F1 Score : 0.6142149060990642


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [06:24:57] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



XGBoost (Tuned):
Best Params: {'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.1}
Accuracy : 0.7020825979527003
Precision: 0.7036266919493671
Recall   : 0.7020825979527003
F1 Score : 0.7014511795418881

===== CB Technique: BioHash =====

RandomForest (Tuned):
Best Params: {'max_depth': None, 'n_estimators': 200}
Accuracy : 0.9459936463113308
Precision: 0.9457517572345606
Recall   : 0.9459936463113308
F1 Score : 0.9455173153129854

SVM (Tuned):
Best Params: {'C': 10, 'gamma': 0.01, 'kernel': 'rbf'}
Accuracy : 0.9527003176844334
Precision: 0.9525813849439089
Recall   : 0.9527003176844334
F1 Score : 0.9525856358746135

KNN (Tuned):
Best Params: {'n_neighbors': 3}
Accuracy : 0.9346981997882103
Precision: 0.9343288285605307
Recall   : 0.9346981997882103
F1 Score : 0.9343838859409022


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [06:32:27] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



XGBoost (Tuned):
Best Params: {'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.1}
Accuracy : 0.9315213554535827
Precision: 0.9310842696516738
Recall   : 0.9315213554535827
F1 Score : 0.930950770678508

===== CB Technique: RP =====

RandomForest (Tuned):
Best Params: {'max_depth': None, 'n_estimators': 200}
Accuracy : 0.9431697846805507
Precision: 0.9427890263803806
Recall   : 0.9431697846805507
F1 Score : 0.9428087379286247

SVM (Tuned):
Best Params: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
Accuracy : 0.9491704906459584
Precision: 0.9489841666523495
Recall   : 0.9491704906459584
F1 Score : 0.9489978200120638

KNN (Tuned):
Best Params: {'n_neighbors': 3}
Accuracy : 0.939286974938228
Precision: 0.9388421334872453
Recall   : 0.939286974938228
F1 Score : 0.9389696391938848


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [06:40:12] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



XGBoost (Tuned):
Best Params: {'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.1}
Accuracy : 0.935757147899753
Precision: 0.9354923894253936
Recall   : 0.935757147899753
F1 Score : 0.9354133299756803


## Recorded results

The saved output above reports an earlier run on 14,161 images across four folder-defined classes. It is retained for provenance and was not rerun after the reproducibility cleanup. The dataset is unavailable in this repository, so the figures have not been independently verified.

| Transformation | Random Forest | SVM | KNN | XGBoost |
|---|---:|---:|---:|---:|
| IoM hashing | 75.11% | 63.93% | 61.63% | 70.21% |
| BioHashing | 94.60% | **95.27%** | 93.47% | 93.15% |
| Random projection | 94.32% | 94.92% | 93.93% | 93.58% |

The highest saved value is 95.27% test accuracy (95.26% weighted F1) for BioHashing with an RBF SVM.


## Limitations and responsible interpretation

- No diffusion model is implemented.
- The dataset source, license, class definitions, subject identities, and collection protocol are not documented.
- The evaluation measures four-class classification, not biometric verification; FAR, FRR, EER, unlinkability, irreversibility, and revocability are not evaluated.
- The fixed transform seed is for repeatability, not a secret or user-specific key. It does not establish template security.
- The 512-unit Dense layer is not trained and therefore acts as a seeded, randomly initialized nonlinear feature projection; Dropout is inactive during `predict`.
- OpenCV supplies BGR images and this retained pipeline does not explicitly convert them to RGB. Fixing that requires a new experimental run.
- The same test set is used to compare many model/transform combinations, so selecting the largest test result can introduce selection bias.
- Dependencies are unpinned, and there is no exported model or standalone inference application.


### Historical Drive-mount cell

The final cell's saved output is retained from the original notebook. Its source now skips the Drive mount outside Colab, but the cell remains redundant because the experiment cell already mounts Drive when the configured dataset path requires it.


In [ ]:
try:
    from google.colab import drive
except ImportError:
    print("Not running in Google Colab; Drive mount skipped.")
else:
    drive.mount("/content/drive")

Mounted at /content/drive
